Here's a **reference notebook** that captures the three async patterns (`await`, `gather`, `create_task`) along with all the insights we’ve uncovered. I’ll annotate the code heavily with comments so you can revisit it later and immediately recall the reasoning.

---

```python
import asyncio

# A fake upload function to simulate network I/O (like uploading to S3).
# We use asyncio.sleep() to mimic waiting on the network.
async def fake_upload(name, delay):
    print(f"Starting {name}")
    await asyncio.sleep(delay)
    print(f"Finished {name}")


# 1️⃣ Sequential with `await`
async def sequential_example():
    print("\n--- Sequential Example ---")
    print("Doing other work before starting uploads...")

    # Each await blocks until the task finishes.
    # Upload2 won't even start until Upload1 is completely done.
    await fake_upload("Upload1", 2)
    await fake_upload("Upload2", 2)

    # Total time ≈ 4 seconds (2 + 2).
    # Printing "Doing other work..." before doesn't change the fact
    # that uploads are strictly sequential.
    print("All done (sequential)")


# 2️⃣ Concurrent with `asyncio.gather`
async def gather_example():
    print("\n--- Gather Example ---")
    print("Doing other work before starting concurrent uploads...")

    # Both uploads start together.
    # The event loop interleaves them while they wait on I/O.
    await asyncio.gather(
        fake_upload("Upload1", 2),
        fake_upload("Upload2", 2)
    )

    # Total time ≈ 2 seconds (the longest task).
    # BUT: you still block at this line until *all* tasks finish.
    print("All done (gather)")


# 3️⃣ Background scheduling with `asyncio.create_task`
async def create_task_example():
    print("\n--- Create Task Example ---")

    # Tasks are scheduled immediately and start running in the background.
    task1 = asyncio.create_task(fake_upload("Upload1", 3))
    task2 = asyncio.create_task(fake_upload("Upload2", 2))

    # While uploads are running, you can do other async work.
    # This is the key difference: uploads are already in progress
    # while you continue with other logic.
    print("Doing other work while uploads run...")
    await asyncio.sleep(1)  # async work overlaps with uploads
    print("Still busy with other work...")

    # Now we decide when to wait for uploads to finish.
    await task1
    await task2

    # Total time ≈ 3 seconds (the longest task).
    # Unlike gather, you had a window to do other work in between.
    print("All done (create_task)")


async def main():
    await sequential_example()
    await gather_example()
    await create_task_example()

asyncio.run(main())
```

---

### 📝 Insights embedded in the code
- **Sequential (`await`)** → blocks at each line, tasks run one after another. Printing early doesn’t change that.  
- **Gather** → starts tasks together, but you still block until *all* finish. Great for parallelism when you want all results at once.  
- **Create Task** → schedules tasks immediately, they run in the background while you continue doing other async work. You choose when to `await` them later.  

⚠️ Important: Async only overlaps **other async work**. If you drop in heavy synchronous code (like a CPU loop), it blocks the event loop and pauses uploads. To mix sync safely, use `asyncio.to_thread` or a process pool.

---
